# 01 Data Cleaning - Hospital Surgeries


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/surgeries.csv")
cleaning_log=[]
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'hospital_surgeries_raw.csv'

## Initial Inspection

In [ ]:
print(df.shape)
display(df.head())
df.info()
display(df.describe(include='all').T)

## Missing Values

In [ ]:
missing=df.isna().sum().to_frame('Missing')
display(missing[missing['Missing']>0])
num=df.select_dtypes(include='number').columns
cat=df.select_dtypes(exclude='number').columns
df[num]=df[num].fillna(df[num].median())
for c in cat:
    if df[c].isna().any():
        df[c]=df[c].fillna(df[c].mode().iloc[0])
cleaning_log.append('Handled missing values')

## Duplicates

In [ ]:
before=len(df)
df=df.drop_duplicates()
print('Removed',before-len(df),'duplicates')
cleaning_log.append('Removed duplicate rows')

## Text Cleaning

In [ ]:
id_cols=['Surgery_ID','Admission_ID','Patient_ID','Doctor_ID','Operation_Theatre_ID']
text=[c for c in df.select_dtypes(include='object').columns if c not in id_cols]
for c in text:
    df[c]=df[c].astype(str).str.strip().str.title()
cleaning_log.append('Standardized text')

## Date Validation

In [ ]:
df['Surgery_Date']=pd.to_datetime(df['Surgery_Date'],errors='coerce')
cleaning_log.append('Validated dates')

## Business Rules

In [ ]:
df=df[df['Duration_Minutes']>0]
df=df[df['Cost']>0]
cleaning_log.append('Applied business rules')

## Feature Engineering

In [ ]:
df['Surgery_Year']=df['Surgery_Date'].dt.year
df['Surgery_Month']=df['Surgery_Date'].dt.month_name()
df['Weekday']=df['Surgery_Date'].dt.day_name()
cleaning_log.append('Created features')

## Data Quality

In [ ]:
quality=pd.DataFrame({
'Metric':['Rows','Columns','Missing Values'],
'Value':[len(df),df.shape[1],int(df.isna().sum().sum())]
})
display(quality)

## Save

In [ ]:
df.to_csv('hospital_surgeries_clean.csv',index=False)
pd.DataFrame({'Cleaning_Step':cleaning_log}).to_csv('cleaning_report.csv',index=False)
print('Files saved.')